# 06. 연결성 정리 — 완전 연결 라우팅 그래프 (그래프 전용)

## 이 노트북이 하는 일
M2 그래프에서 **왕복 불가 구간(일방통행 stub 등)** 을 제거해 **최대 강연결요소(SCC)** 만 남기고, 병원·안전센터·발생지를 다시 스냅한다. 결과 `graph_drive_conn` 은 이후 **집계구 라인(10·11)** 이 도달지연 계산에 사용한다.

## 왜 이렇게 설계했나 (설계 이유)
- **왜 강연결(SCC)인가:** 구급차는 '가서(출동)' + '와야(이송)' 한다 = 양방향. 강연결요소는 임의 두 노드가 서로 오갈 수 있는 최대 덩어리.
- **왜 왜 제거하나:** M2엔 SCC가 여러 개(일방통행·경계 잘림 stub). 이런 노드에 스냅하면 Dijkstra가 경로를 못 찾음.
- **왜 재스냅하나:** SCC만 남기면 노드 집합이 바뀜 → 병원·센터·발생지를 새 노드집합의 최근접으로 다시 붙임. (10·11이 `node_type=="station"` 라벨로 안전센터를 찾으므로 재스냅 필수)
- **왜 그래프 전용인가(개정):** 이전 버전은 100m 격자(04·05)도 재스냅해 `grid_risk_conn`을 만들었으나, **현재 분석은 집계구(10·11)** 라 100m 격자는 쓰지 않는다. 불필요한 의존을 떼어 이 노트북을 자립화했다. (구 100m 라인은 archive 참조)

## 데이터 출처
- 그래프: 03(M2, OSM+SRTM).

In [ ]:
import os, warnings                       # 폴더·경고
warnings.filterwarnings("ignore")
import numpy as np                          # 좌표 배열
import networkx as nx, osmnx as ox, geopandas as gpd   # 그래프·로드·지리표
CRS_M=5186                                  # 평면좌표계
def _sf(x):                                 # M2의 빈 문자열 속성을 실수로 안전 변환
    try: return float(x)
    except: return float("nan")
G=ox.load_graphml("outputs/graph_drive_M2.graphml",                                  # 03의 M2 그래프 로드
    edge_dtypes={"grade":_sf,"grade_abs":_sf,"elev_u":_sf,"elev_v":_sf,"width_est":_sf,"length":float},
    node_dtypes={"elev":_sf})
print("원본: 노드", G.number_of_nodes(), "엣지", G.number_of_edges(),
      "| SCC", nx.number_strongly_connected_components(G))                            # 정리 전 강연결요소 개수

## 1. 특수노드 좌표 보존 → 최대 SCC 추출

In [ ]:
specials={}                                                                           # 병원·센터·발생지 '좌표' 미리 저장(재스냅용)
for n,d in G.nodes(data=True):
    t=d.get("node_type")
    if t in ("hospital","station","incident"):
        specials.setdefault(t,[]).append((G.nodes[n]["x"], G.nodes[n]["y"], d.get("snap_ref","")))

Gc = ox.truncate.largest_component(G, strongly=True)                                  # 최대 강연결요소만 남긴 새 그래프
print("정리 후: 노드", Gc.number_of_nodes(), "엣지", Gc.number_of_edges(),
      "| SCC", nx.number_strongly_connected_components(Gc),                           # 1이어야 정상
      "| 제거 노드", G.number_of_nodes()-Gc.number_of_nodes())

## 2. 특수노드 재스냅 + 안전센터 거리 필터 (10·11이 이 라벨 사용)

**출동 범위 제한:** OSM fire_station을 넓은 graph_area에서 받아 인근 구 시설까지 섞임(최대 3km). 동구 심정지에 3km 밖 출동은 비현실적 → **대상지(초량·좌천)에서 1.5km 이내 안전센터만** 출발지로 남긴다.

In [ ]:
import geopandas as gpd
from shapely.geometry import box
from shapely.ops import unary_union
# 대상지(초량·좌천) 경계 — 안전센터 거리 필터용
adm=ox.features_from_polygon(box(129.020,35.100,129.075,35.155), tags={"boundary":"administrative"})
adm=adm[adm.geometry.geom_type.isin(["Polygon","MultiPolygon"])]; adm["name"]=adm["name"].astype(str)
studym=gpd.GeoSeries([unary_union(adm[adm["name"].str.contains("초량|좌천",na=False)].geometry)],crs=4326).to_crs(5186).iloc[0]
STATION_MAX_M = 1500                                                                  # 대상지에서 1.5km 이내만 출발지 인정

for n,d in Gc.nodes(data=True):                                                       # 노드 유형 초기화
    d["node_type"]="road"; d["snap_ref"]=""
snap={}
for t,pts in specials.items():                                                        # 저장해둔 좌표로 다시 스냅
    xs=[p[0] for p in pts]; ys=[p[1] for p in pts]; refs=[p[2] for p in pts]
    nn=ox.distance.nearest_nodes(Gc, X=xs, Y=ys)                                       # 새 노드집합에서 최근접
    nn=nn if isinstance(nn,(list,np.ndarray)) else [nn]
    kept=[]
    for nid,ref,px,py in zip(nn,refs,xs,ys):
        if t=="station":                                                              # 안전센터는 거리 필터 적용
            from shapely.geometry import Point
            if Point(px,py).distance(studym) > STATION_MAX_M:                          # 대상지에서 1.5km 초과면
                continue                                                              # 출발지에서 제외(먼 인근구 시설)
        if Gc.nodes[nid]["node_type"]=="road":
            Gc.nodes[nid]["node_type"]=t; Gc.nodes[nid]["snap_ref"]=ref               # 유형·이름 부여
            kept.append(nid)
    snap[t]=list(dict.fromkeys(int(x) for x in kept))
for t,ns in snap.items():
    extra = f" (1.5km 필터 적용)" if t=="station" else ""
    print(f"  {t}: {len(ns)}개 노드{extra}")

## 3. 검증 + 저장 (그래프만)

In [ ]:
assert nx.number_strongly_connected_components(Gc)==1, "아직 완전 연결 아님"           # 강연결요소 1개 강제 확인
print("✅ 검증 통과: 강연결요소 1개 (어디서든 왕복 가능)")

ox.save_graphml(Gc, "outputs/graph_drive_conn.graphml")                               # ★ 기준 라우팅 그래프 저장(10·11이 읽음)
try: ox.save_graph_geopackage(Gc, "outputs/graph_drive_conn.gpkg")                    # QGIS용
except Exception as e: print("gpkg 경고:", e)

def stringify(gdf):                                                                   # parquet 저장용 타입 정리
    g = gdf.copy()
    for c in g.columns:
        if c=="geometry": continue
        if g[c].apply(lambda v: isinstance(v,(list,tuple))).any():
            g[c]=g[c].apply(lambda v: ";".join(map(str,v)) if isinstance(v,(list,tuple)) else v)
        if g[c].dtype==object: g[c]=g[c].astype(str)
    return g
ng,eg=ox.graph_to_gdfs(Gc)
for g,nm in [(ng,"nodes"),(eg,"edges")]:
    try: stringify(g).to_parquet(f"outputs/{nm}_drive_conn.parquet")
    except Exception as e: print(nm,"parquet 경고:",e)
print("저장:", [f for f in sorted(os.listdir("outputs")) if "conn" in f])

## 4. 연결성 확인 지도 (제거된 노드 표시)

In [ ]:
import folium                                                                          # 지도
dropped=[n for n in G.nodes if n not in Gc.nodes]                                     # 제거된 노드(왕복 불가 stub)
dp=gpd.GeoSeries(gpd.points_from_xy([G.nodes[n]["x"] for n in dropped],
                                    [G.nodes[n]["y"] for n in dropped]),crs=CRS_M).to_crs(4326)
edges=ox.graph_to_gdfs(Gc, nodes=False)[["geometry"]].to_crs(4326)                   # 연결 도로망
station_nodes=[n for n,d in Gc.nodes(data=True) if d.get("node_type")=="station"]      # 재스냅된 센터
m=folium.Map(location=[35.122,129.045], zoom_start=13, tiles="cartodbpositron")
folium.GeoJson(edges.to_json(), name="연결 도로망",
               style_function=lambda x:{"color":"#9ecae1","weight":1}).add_to(m)
for p in dp:                                                                          # 제거된 노드(빨강) — 전부 가장자리인지 확인
    folium.CircleMarker([p.y,p.x], radius=3, color="#d7191c", fill=True, fillOpacity=0.9,
                        tooltip="제거(왕복불가)").add_to(m)
st=gpd.GeoSeries(gpd.points_from_xy([Gc.nodes[n]["x"] for n in station_nodes],
                                    [Gc.nodes[n]["y"] for n in station_nodes]),crs=CRS_M).to_crs(4326)
for p in st:
    folium.CircleMarker([p.y,p.x], radius=5, color="blue", fill=True, tooltip="119안전센터").add_to(m)
folium.LayerControl().add_to(m)
m.save("outputs/connectivity_map.html")                                               # 연결성 지도 저장
print("지도 저장: outputs/connectivity_map.html (빨강=제거된 경계 stub)")
m